In [ ]:
import os
# Force numpy/BLAS to use fewer threads to prevent CPU thrashing
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"

import csv
import time
import numpy as np
import h5py

import tenpy
from tenpy.networks.site import SpinHalfFermionSite
from tenpy.models.lattice import Kagome
from tenpy.models.model import CouplingMPOModel
from tenpy.networks.mps import MPS
from tenpy.algorithms import dmrg
from tenpy.tools.hdf5_io import save

from tenpy.tools.hdf5_io import load, load_from_hdf5

import warnings

warnings.filterwarnings("ignore", message = "H is zero in the given block")
warnings.filterwarnings("ignore", message = f"unused option \['N_elec'\]")

# Ensure output directories exist
os.makedirs("data_outputs", exist_ok=True)
os.makedirs("data_outputs/wavefunctions", exist_ok=True)

# ---------------------------------------------------------
# 1. Custom Kagome Hubbard Model
# ---------------------------------------------------------
class KagomeHubbardModel(CouplingMPOModel):
    """
    Custom Kagome Fermi-Hubbard model supporting NNN hopping.
    """
    def init_sites(self, model_params):
        # SpinHalfFermionSite uses separate kwargs for particle and spin conservation
        cons_N = model_params.get('cons_N', 'N')
        cons_Sz = model_params.get('cons_Sz', 'Sz')
        return SpinHalfFermionSite(cons_N=cons_N, cons_Sz=cons_Sz)

    def init_lattice(self, model_params):
        Lx = model_params.get('Lx', 1)
        Ly = model_params.get('Ly', 1)
        bc_x = model_params.get('bc_x', 'periodic')
        bc_y = model_params.get('bc_y', 'periodic')
        site = self.init_sites(model_params)
        kagome_model = Kagome(Lx, Ly, site, bc=[bc_x, bc_y])
        return kagome_model

    def init_terms(self, model_params):
        #nearest-neighbor hopping, next-nearest-neighbor hopping, and coulomb interaction
        t1 = model_params.get('t1', 1.0)
        t2 = model_params.get('t2', 0.0)
        U = model_params.get('U', 0.0)
        
        # Site-dependent chemical potential (array of size N_sites)
        # Defaults to None, meaning 0.0 for all sites
        mu_array = model_params.get('mu', None)

        #TABC
        theta_x = model_params.get('theta_x', 0.0)
        theta_y = model_params.get('theta_y', 0.0)
        # Nearest-neighbor hopping
        for u1, u2, dx in self.lat.pairs['nearest_neighbors']:
            #TABC Phase
            phase = np.exp(1j * (theta_x * dx[0] + theta_y * dx[1]))
            
            self.add_coupling(-t1*phase, u1, 'Cdu', u2, 'Cu', dx, plus_hc=True)
            self.add_coupling(-t1*phase, u1, 'Cdd', u2, 'Cd', dx, plus_hc=True)

        # Next-nearest-neighbor hopping
        for u1, u2, dx in self.lat.pairs['next_nearest_neighbors']:
            #TABC Phase
            phase = np.exp(1j * (theta_x * dx[0] + theta_y * dx[1]))
            
            self.add_coupling(-t2*phase, u1, 'Cdu', u2, 'Cu', dx, plus_hc=True)
            self.add_coupling(-t2*phase, u1, 'Cdd', u2, 'Cd', dx, plus_hc=True)

        # On-site Coulomb interaction
        for u in range(len(self.lat.unit_cell)):
            self.add_onsite(U, u, 'NuNd')

        # ---------------------------------------------------------
        # SITE-DEPENDENT CHEMICAL POTENTIAL
        # ---------------------------------------------------------
        if mu_array is not None:
            # Validate array length matches total lattice sites
            if len(mu_array) != self.lat.N_sites:
                raise ValueError(f"Length of 'mu' array ({len(mu_array)}) must match total lattice sites ({self.lat.N_sites}).")
            
            # 1. Create a spatial array to match the Kagome lattice geometry
            # Shape: (Lx, Ly, num_unit_cell_sites)
            Lx, Ly = self.lat.Ls
            mu_spatial = np.zeros((Lx, Ly, len(self.lat.unit_cell)))
            
            # 2. Map the 1D MPS array into the 3D lattice array
            for i in range(self.lat.N_sites):
                # lat.mps2lat_idx(i) returns the lattice coordinates (x, y, u)
                x, y, u = self.lat.mps2lat_idx(i)
                mu_spatial[x, y, u] = mu_array[i]
                
            # 3. Apply the spatially-dependent array to each unit cell site (u)
            for u in range(len(self.lat.unit_cell)):
                # mu_spatial[:, :, u] extracts the (Lx, Ly) array of strengths for site 'u'
                # -mu * (n_up + n_dn)
                self.add_onsite(-mu_spatial[:, :, u], u, 'Ntot')
# ---------------------------------------------------------
# 2. Helpers for Initial States & Parameters
# ---------------------------------------------------------
def generate_product_state(N, N_elec):
    """
    Distributes electrons to match total N_elec while minimizing Sz.
    Creates a valid product state string for SpinHalfFermionSite.
    """
    state = ['empty'] * N
    N_up = N_elec // 2 + (N_elec % 2)
    N_down = N_elec // 2
    
    # Fill spin-ups from the left
    for i in range(N_up):
        state[i] = 'up'
    # Fill spin-downs from the right (allows double occupancy 'full' if needed)
    for i in range(N_down):
        idx = N - 1 - i
        if state[idx] == 'empty':
            state[idx] = 'down'
        else:
            state[idx] = 'full'
    return state

def get_random_U():
    """Stratified sampling for U."""
    block = np.random.choice(['weak', 'intermediate', 'strong'])
    if block == 'weak': return np.random.uniform(0.1, 2.0)
    elif block == 'intermediate': return np.random.uniform(2.0, 6.0)
    else: return np.random.uniform(6.0, 10.0)
def get_kagome_positions(model_params, L):
    """
    Reconstruct the Kagome lattice geometry from saved model_params to extract exact 2D spatial coordinates of each site
    """
    cons_N = model_params.get('cons_N', 'N')
    cons_Sz = model_params.get('cons_Sz', 'Sz')
    site = SpinHalfFermionSite(cons_N=cons_N, cons_Sz=cons_Sz)
    Lx = model_params['Lx']
    Ly = model_params['Ly']
    bc_x = model_params.get('bc_x', 'periodic')
    bc_y = model_params.get('bc_y', 'periodic')

    lattice = Kagome(Lx, Ly, site, bc=[bc_x, bc_y])

    #extract physical 2D coordinates for all L sites
    return np.array([lattice.position(lattice.mps2lat_idx(i)) for i in range(L)])

def calculate_structure_factor(correlation_matrix, positions, q_vectors):
    """
    Compute S(q) for (1/N) * sum_{j, l} exp (i * q  DOT (r_j - r_l)) * C{jl}
        1/N * Summation (j,l): ( Phase Matrix * Correlation Matrix)
    """
    N = correlation_matrix.shape[0]
    S_q = np.zeros(len(q_vectors), dtype=complex)
    #print('begin idx,q loop')
    for idx, q in enumerate(q_vectors):
        #Calculate distance vectors between all pairs of sites (r_j - r_l)
        #np.newaxis adds a new col/row to its position (:, np.newaxis, :) = (X, Y+1, Z)...
        #In this case, positions.shape = (X+N, Y+N, Z)
        r_diff = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]
        #print('\t got r_diff')
        #calculate phase matrix: exp(i*q DOT r_diff)
        phase_matrix = np.exp(1j * np.dot(r_diff, q))
        #print('\t got phase matrix')
        #Multiple element-wise with the correlation matrix and sum
        S_q[idx] = np.sum(phase_matrix * correlation_matrix)/N
        #print(f'\t got S_q[{idx}]')
    #return the real aspects of S_q
    return np.real(S_q)
        
def measure_properties(psi, model_params, **kwargs):
    #print(f"measuring properties...")
    #1.  Calculate basic observables mapping out charge and spin densities
    charge_density = psi.expectation_value('Ntot') # Total particles per site
    spin_density = psi.expectation_value('Sz')     # Sz per site
    double_occupancy = psi.expectation_value('NuNd') 
    #print("pass charge density, spin density, double occupancy")
    # 2. Compute Heavy Correlation Functions
    # Calculate Spin-Spin correlation <Sz_i Sz_j> across the whole lattice
    # Using term_correlation_function_right for optimal performance
    
    spin_corr_matrix = psi.correlation_function('Sz', 'Sz')
    
    #spin_corr_matrix = psi.term_correlation_function_right([('Sz', 0)],[('Sz', 1)])
    #print(f'spin_corr_matrix: {spin_corr_matrix}')
    #print(f'shape: {spin_corr_matrix.shape}')
    #print("pass spin correlation matrix")
    # Calculate Charge-Charge correlation <N_i N_j>

    charge_corr_matrix = psi.correlation_function('Ntot', 'Ntot')
    
    #charge_corr_matrix = psi.term_correlation_function_right([('Ntot', 0)],[('Ntot', 1)])
    #print("pass charge correlation matrix")
    # Calculate Structure Factors (Requires Fourier transforming the correlation matrix)
    # Example: Simple sum of spin-spin correlations as a scalar placeholder
    #S_q_zero = np.sum(spin_corr_matrix)

    #Structure Factors: reconstruct site positions
    positions = get_kagome_positions(model_params, psi.L)
    #print("pass position kagome")
    #define momentum vectors from -pi to pi
    #ADJUST DENSITY
    momentum_density = kwargs.get("momentum_density", 20)
    qx = np.linspace(-np.pi, np.pi, momentum_density)
    qy = np.linspace(-np.pi, np.pi, momentum_density)
    QX, QY = np.meshgrid(qx, qy)
    q_vectors = np.column_stack([QX.ravel(), QY.ravel()])
    #print("pass q vectors")
    #Spin Structure Factor:
    S_q_spin = calculate_structure_factor(spin_corr_matrix, positions, q_vectors)
    S_q_charge = calculate_structure_factor(charge_corr_matrix, positions, q_vectors)
    #print(f"meaured properties")
    return {
        "charge_density": charge_density,
        "double_occupancy": double_occupancy,
        "spin_corr": spin_corr_matrix,
        "charge_corr": charge_corr_matrix,
        "q_vectors": q_vectors,
        "S_q_spin": S_q_spin,
        "S_q_charge": S_q_charge
    }

def load_and_measure(run_id, h5_filepath, **kwargs):
    """
    Loads a highly optimized MPS and computes custom correlation matrices.
    """
    print(f"Loading {run_id}...")
    
    # 1. Reconstruct MPS from HDF5
    # TeNPy's load function takes the string path directly
    #data = load(h5_filepath)
    
    #psi = data['psi']
    #model_params = data['model_params']
    with h5py.File(h5_filepath, 'r') as f_h5:
        #data = load(f_h5)
        psi = load_from_hdf5(f_h5, 'psi')
        model_params = load_from_hdf5(f_h5, 'model_params')
    
    # Calculate basic observables mapping out charge and spin densities
    charge_density = psi.expectation_value('Ntot') # Total particles per site
    spin_density = psi.expectation_value('Sz')     # Sz per site
    double_occupancy = psi.expectation_value('NuNd') 

    # 2. Compute Heavy Correlation Functions
    # Calculate Spin-Spin correlation <Sz_i Sz_j> across the whole lattice
    # Using term_correlation_function_right for optimal performance
    spin_corr_matrix = psi.term_correlation_function_right([('Sz', 0)],[('Sz', 1)])
    
    # Calculate Charge-Charge correlation <N_i N_j>
    charge_corr_matrix = psi.term_correlation_function_right([('Ntot', 0)],[('Ntot', 1)])
    
    # Calculate Structure Factors (Requires Fourier transforming the correlation matrix)
    # Example: Simple sum of spin-spin correlations as a scalar placeholder
    #S_q_zero = np.sum(spin_corr_matrix)

    #Structure Factors: reconstruct site positions
    positions = get_kagome_positions(model_params, psi.L)

    #define momentum vectors from -pi to pi
    #ADJUST DENSITY
    momentum_density = kwargs.get("momentum_density", 20)
    qx, qy = np.linspace(-np.pi, np.pi, momentum_density)
    QX, QY = np.meshgrid(qx, qy)
    q_vectors = np.column_stack([QX.ravel(), QY.ravel()])

    #Spin Structure Factor:
    S_q_spin = compute_structure_factor(spin_corr_matrix, positions, q_vectors)
    S_q_charge = compute_structure_factor(charge_corr_matrix, positions, q_vectors)
    
    
    # 3. Export Data for the AI Agent
    # In a real workflow, you would save these arrays directly into standard .npy files
    print(f"--- Analysis for {run_id} ---")
    print(f"System Size: {psi.L} sites")
    print(f"Average Double Occupancy: {np.mean(double_occupancy):.4f}")
    print(f"S_q Spin Structure Factor: {S_q_spin}\n")
    print(f"S_q Charge Structure Factor: {S_q_charge}\n")
    print(f"Model Parameters: {model_params}\n")

    print(f"Appending observables to {run_id}.h5")

    #open file in append mode
    '''
    with h5py.File(h5_filepath, 'a') as f_h5:
        if 'observables' in f_h5:
            print(f"observables already exist: delete and re-append")
            del f_h5['observables']

        obs_group = f_h5.create_group('observables')
        obs_group['spin_corr_matrix'] = spin_corr_matrix
        obs_group['charge_corr_matrix'] = charge_corr_matrix
        obs_group['charge_density'] = charge_density
        obs_group['double_occupancy'] = double_occupancy
        obs_group['q_vectors'] = q_vectors #shape (400, 2) 400 dependant on momentum density
        obs_group['S_q_spin'] = S_q_spin #shape (400, x)
        obs_group['S_q_charge'] = S_q_charge #shape(400, x)
    print(f"\tAppended!\n")
    '''
    return {
        "charge_density": charge_density,
        "double_occupancy": double_occupancy,
        "spin_corr": spin_corr_matrix,
        "charge_corr": charge_corr_matrix,
        "S_q_spin": S_q_spin,
        "S_q_charge": S_q_charge
    }
def apply_geometric_symmetrization(observables, model_params, psi) -> dict:
    """
    Applies real-space geometric symmetrization to local observables and 
    correlation matrices based on lattice site distances.
    """
    site_positions = get_kagome_positions(model_params, psi.L)
    num_sites = len(site_positions)
    symmetrized = observables.copy()  # Preserve keys we don't modify (like q_vectors)

    # 1. Symmetrize Local Observables (Site-Averaging)
    # Checks for local phenomenon
    local_keys = ['charge_density', 'double_occupancy']
    
    for key in local_keys:
        if key in observables and observables[key] is not None:
            val_array = np.array(observables[key])
            avg_val = np.mean(val_array)
            symmetrized[key] = [float(avg_val)] * num_sites

    # 2. Symmetrize Correlation Matrices (Distance-Averaging)
    matrix_keys = ['spin_corr', 'charge_corr']              # matrix names
    
    # Pre-compute distance groups for the given geometry
    distance_groups = {}
    for i in range(num_sites):
        for j in range(num_sites):
            dist = np.linalg.norm(site_positions[i] - site_positions[j])
            # Round distance to 5 decimals to handle floating point noise
            dist_key = np.round(dist, decimals=5)
            
            if dist_key not in distance_groups:
                distance_groups[dist_key] = []
            distance_groups[dist_key].append((i, j))

    # Apply distance-averaging to any matrices found in the dictionary
    for key in matrix_keys:
        if key in observables and observables[key] is not None:
            matrix = np.array(observables[key])
            sym_matrix = np.zeros_like(matrix)
            
            for dist_key, pairs in distance_groups.items():
                # Find the mean correlation for all site pairs at this exact distance
                avg_val = np.mean([matrix[i, j] for i, j in pairs])
                
                # Assign the averaged value back to all corresponding pairs
                for i, j in pairs:
                    sym_matrix[i, j] = float(avg_val)
                    
            symmetrized[key] = sym_matrix.tolist()
            
    return symmetrized


In [ ]:
# ---------------------------------------------------------
# 3. Main Generation Loop
# ---------------------------------------------------------
def main():
    num_samples = 2
    # Lx, Ly tuples matching N = 3, 6, 9, 12, 15 (Kagome has 3 sites/unit cell)
    geometries = [(1,1), (2,1), (3,1), (2,2), (4,1)]
    filling_targets = [0.333, 0.417, 0.5, 0.8, 1.0, 1.2]
    
    #csv_file = "data_outputs/master_index.csv"
    #csv_file = "data_outputs/three_sites_index.csv"
    #csv_file = "data_outputs/six_sites_index.csv"
    csv_file = "data_outputs/Tenpy_Outputs1/twentyseven_sites_index.csv"
    #csv_file = "data_outputs/twelve_sites_index.csv"
    
    # Initialize CSV
    with open(csv_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Run_ID", "Lx", "Ly", "N_sites", "t1", "t2", "U", 
                         "Target_Filling", "N_elec", "Energy", "Max_Err", 
                         "Converged", "Duration_sec"])

    print(f"Starting dataset generation of {num_samples} configurations...")

    for i in range(num_samples):
        run_id = f"Kagome_Hubbard_{i:03d}"
        print(f"\n--- Running {run_id} ---")
        
        # Sample parameters
        Lx, Ly = geometries[np.random.choice(len(geometries))]

        #CHOOSE SPECIFIC GEOMETRIES
        #Lx, Ly = (1,1)
        #Lx, Ly = (2,1)
        #Lx, Ly = (3,1)
        Lx, Ly = (3,3)
        
        N_sites = 3 * Lx * Ly
        t1 = np.random.uniform(-0.3, 0.3)
        #t2 = np.random.uniform(-0.3, 0.3)
        t2 = 0.0
        U = get_random_U()
        #weak/moderate interactions (U/t2 < 6.0)
        #while U/t2 >= 6.0:
        #    U = get_random_U()
        filling = np.random.choice(filling_targets)
        N_elec = max(1, min(int(round(filling * N_sites)), 2 * N_sites))
        
        model_params = {
            'Lx': Lx, 'Ly': Ly,
            'bc_x': 'periodic', 'bc_y': 'periodic',
            't1': t1, 't2': t2, 'U': U,
            'cons_N': 'N',     # Conservations split for SpinHalfFermionSite
            'cons_Sz': 'Sz'
        }
        
        start_time = time.time()
        energy, max_err = np.nan, np.nan
        converged = False
        
        try:
            # 1. Build Model
            model = KagomeHubbardModel(model_params)
            
            # 2. Build Initial State
            init_str = generate_product_state(N_sites, N_elec)
            psi = MPS.from_product_state(model.lat.mps_sites(), init_str, model.lat.bc_MPS, unit_cell_width=model.lat.N_sites)
            
            # 3. Configure DMRG (CPU optimized)
            dmrg_params = {
                'mixer': True,
                'max_E_err': 1.0,
                'trunc_params': {
                    'chi_max': 800, # Conservative bound for laptop memory
                    'svd_min': 1e-10
                },
                'norm_tol': 1e-3,
                'chi_list': {
                    0:100,
                    10:200,
                    20:400,
                    30:800
                },
                'max_sweeps': 100, # Allows time for long-range PBC correlations to converge
                'min_sweep':20,
                'combine':True
            }
            
            # 4. Run DMRG Engine
            eng = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
            energy, max_err = eng.run()
            converged = True
            
            # 5. Serialize Wavefunction
            # Utilizing TeNPy's HDF5 wrapper ensures proper tensor reconstruction ADDING ENERGY TO THE LIST
            h5_path = f"data_outputs/Tenpy_Outputs1/twentyseven_site_{run_id}.h5"
            #with h5py.File(h5_path, 'w') as f_h5:
                #save(f_h5, {'psi': psi, 'model_params': model_params})
            #save({'psi': psi, 'model_params': model_params, 'energy':energy}, h5_path)
            obs = {};
            obs = load_and_measure(run_id, h5_path);
            save({'U':U, 't1':t1, 't2':t2, 'filling':filling}, h5_path)
            with h5py.File(h5_filepath, 'a') as f:
                #f.create_dataset('U', data=U)
                #f.create_dataset('t1', data=t1)
                #f.create_dataset('t2', data=t2)
                #f.create_dataset('filling', data=filling)
                f.create_dataset('L', data=Lx)
                f.create_dataset('Lx', data=Lx)
                f.create_dataset('Ly', data=Ly)
                f.create_dataset('n', data=obs["charge_density"])
                f.create_dataset('d', data=obs["double_occupancy"])
                f.create_dataset('C', data=obs["spin_corr"])
                f.create_dataset('C2', data=obs["charge_corr"])
                #f.create_dataset('B', data=B)
                #f.create_dataset('Adj1', data=Adj1)
                #f.create_dataset('Adj2', data=Adj2)
                f.create_dataset('energy', data=energy)
                f.create_dataset('S_q_spin', data=obs["S_q_spin"])
                f.create_dataset('S_q_charge', data=obs["S_q_charge"])

            print("Observables added to file")
                
                
        except Exception as e:
            print(f"Warning: Configuration {run_id} failed. Reason: {e}")
            converged = False
            
        duration = time.time() - start_time
        print(f"Result: E={energy:.6f}, N={N_sites}, N_elec={N_elec}, U={U:.2f} | Time: {duration:.1f}s")
        
        # 6. Save Metadata
        with open(csv_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([run_id, Lx, Ly, N_sites, t1, t2, U, 
                             filling, N_elec, energy, max_err, 
                             converged, duration])

if __name__ == "__main__":
    main()

In [15]:
"""
================================================================================
 kagome_cnn.py  --  size-agnostic, multi-task CNN for the kagome Hubbard model
================================================================================

WHY A CNN AND NOT A GNN
-----------------------
Kagome is NOT an arbitrary graph. It is a triangular Bravais lattice with a
3-site basis (sublattices A, B, C), so it maps exactly onto a tensor

        (L, L, 3)      L x L unit cells,  3 sublattice channels

Every kagome bond fits inside a 3x3 kernel window (verified: cell offsets are
only -1, 0, +1 in each direction).

A message-passing GNN aggregates over a site's neighbours in a
permutation-invariant way, which DISCARDS which neighbour is which. On kagome
the up-triangle and down-triangle bonds are geometrically distinct but identical
as graph edges, so the GNN sees {same,same,same,same} and the frustration
structure is destroyed at the first layer. (Formally: kagome and the square
lattice are both 4-regular, hence indistinguishable by 1-WL with uniform node
features; and message passing provably cannot count triangles.)

A convolution kernel does not throw that away. Kernel position (-1,0) is a
DIFFERENT learnable weight from (+1,0), so both triangles are visible for free.

SIZE-AGNOSTICISM
----------------
Comes free IF the network is fully convolutional. No dense layers anywhere.
Use mean pooling (not sum) for global quantities. Then the same weights run on
L=2, 4, 8 ... unit cells.

WHAT THIS FILE CONTAINS
-----------------------
  1. Kagome lattice geometry + bond table            (pure numpy, testable)
  2. Synthetic data generator (physics-flavoured)    -- so you can run TODAY,
                                                        before ED data exists
  3. The model                                       (PyTorch)
  4. Multi-task loss with hard-wired physics
  5. Baselines you MUST beat before believing anything
  6. Sanity checks against known kagome values

Requires: numpy, torch.   Run:  python kagome_cnn.py
================================================================================
"""

import numpy as np
import time
import h5py
import gc

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAVE_TORCH = True
except ImportError as I:
    HAVE_TORCH = False
    print("[warn] torch not found -- geometry and baseline sections will still run")
    print(I)


# =============================================================================
# 1. KAGOME GEOMETRY
# =============================================================================

A1 = np.array([1.0, 0.0])                    # Bravais vectors (triangular)
A2 = np.array([0.5, np.sqrt(3) / 2])
BASIS = {0: np.array([0.0, 0.0]),            # A
         1: A1 / 2,                          # B
         2: A2 / 2}                          # C

# The 12 directed bonds, as (d_row, d_col, sublattice_from, sublattice_to).
# Verified by distance search: these are ALL nearest-neighbour bonds, degree 4.
BOND_TABLE = [
    (-1,  0, 0, 2), (-1, +1, 1, 2),
    ( 0, -1, 0, 1), ( 0,  0, 0, 1), ( 0,  0, 0, 2),
    ( 0,  0, 1, 0), ( 0,  0, 1, 2), ( 0,  0, 2, 0), ( 0,  0, 2, 1),
    ( 0, +1, 1, 0), (+1, -1, 2, 1), (+1,  0, 2, 0),
]

'''
def site_positions(L):
    """Cartesian position of every site. Returns array of shape (L, L, 3, 2)."""
    pos = np.zeros((L, L, 3, 2))
    for r in range(L):
        for c in range(L):
            for s in range(3):
                pos[r, c, s] = r * A2 + c * A1 + BASIS[s]
    return pos


def flat_index(r, c, s, L):
    """Map (row, col, sublattice) -> flat site index in [0, 3L^2)."""
    return (r * L + c) * 3 + s


def build_adjacency(L):
    """Nearest-neighbour adjacency list, periodic. Use this to build ED
    Hamiltonians so the NN and the ED data agree on the lattice."""
    adj = {i: [] for i in range(3 * L * L)}
    for r in range(L):
        for c in range(L):
            for (dr, dc, s_from, s_to) in BOND_TABLE:
                i = flat_index(r, c, s_from, L)
                j = flat_index((r + dr) % L, (c + dc) % L, s_to, L)
                if j not in adj[i]:
                    adj[i].append(j)
    return adj

def pair_displacements(L):
    """Minimum-image relative displacement r_ij for every ordered pair.
    Returns (N, N, 2) with N = 3L^2. Used as input to the pair decoder."""
    pos = site_positions(L).reshape(-1, 2)          # (N, 2)
    N = pos.shape[0]
    # lattice vectors of the supercell
    T1, T2 = L * A1, L * A2
    best = np.full((N, N, 2), np.inf)
    best_norm = np.full((N, N), np.inf)
    for m in (-1, 0, 1):
        for n in (-1, 0, 1):
            shift = m * T1 + n * T2
            d = pos[None, :, :] - pos[:, None, :] + shift
            nrm = np.linalg.norm(d, axis=-1)
            take = nrm < best_norm
            best_norm[take] = nrm[take]
            best[take] = d[take]
    return best, best_norm
'''
def site_positions(Lx, Ly):
    """Cartesian position of every site. Returns array of shape (L, L, 3, 2)."""
    pos = np.zeros((Lx, Ly, 3, 2))
    for r in range(Lx):
        for c in range(Ly):
            for s in range(3):
                pos[r, c, s] = r * A2 + c * A1 + BASIS[s]
    return pos
    
def flat_index(r, c, s, Lx):
    """Map (row, col, sublattice) -> flat site index in [0, 3L^2)."""
    return (r * Lx + c) * 3 + s
    
def build_adjacency(Lx, Ly):
    """
    Nearest-neighbour adjacency list with periodic boundary conditions
    for an Lx x Ly Kagome lattice (3 * Lx * Ly total sites).
    """
    num_sites = 3 * Lx * Ly
    adj = {i: [] for i in range(num_sites)}

    for r in range(Ly):         # Row loop bounded by Ly
        for c in range(Lx):     # Column loop bounded by Lx
            for (dr, dc, s_from, s_to) in BOND_TABLE:
                # Source site
                i = flat_index(r, c, s_from, Lx)
                
                # Target site with periodic wrapping along respective axes
                r_target = (r + dr) % Ly
                c_target = (c + dc) % Lx
                j = flat_index(r_target, c_target, s_to, Lx)
                
                if j not in adj[i]:
                    adj[i].append(j)
                    
    return adj
    
def pair_displacements(Lx, Ly, A1=np.array([1.0, 0.0]), A2=np.array([0.5, np.sqrt(3)/2])):
    """
    Minimum-image relative displacement r_ij for every ordered pair on an Lx x Ly Kagome lattice.
    Returns (best, best_norm) with shapes (N, N, 2) and (N, N), where N = 3 * Lx * Ly.
    Used as input to the pair decoder.
    """
    # Extract coordinates for all N = 3 * Lx * Ly sites
    pos = site_positions(Lx, Ly).reshape(-1, 2)          # (N, 2)
    N = pos.shape[0]
    
    # Lattice vectors of the Lx x Ly supercell
    T1 = Lx * A1
    T2 = Ly * A2
    
    best = np.full((N, N, 2), np.inf)
    best_norm = np.full((N, N), np.inf)
    
    # Search over 3x3 periodic image wrap-arounds
    for m in (-1, 0, 1):
        for n in (-1, 0, 1):
            shift = m * T1 + n * T2
            d = pos[None, :, :] - pos[:, None, :] + shift
            nrm = np.linalg.norm(d, axis=-1)
            
            take = nrm < best_norm
            best_norm[take] = nrm[take]
            best[take] = d[take]
            
    return best, best_norm


# =============================================================================
# 2. SYNTHETIC DATA  (placeholder -- replace with ED as soon as it exists)
# =============================================================================
# Physics put in deliberately, so you can verify the whole pipeline works
# BEFORE spending compute on exact diagonalization:
#   - correlations decay exponentially, xi ~ 1 lattice spacing
#   - nearest neighbour NEGATIVE  (target ~ -0.22 in the Heisenberg limit)
#   - second neighbour small POSITIVE
#   - double occupancy falls as U/t grows
# If the model cannot learn THIS, it will not learn ED data either.

def synthetic_sample(L, U, filling, rng):
    rij, dist = pair_displacements(Lx, Ly)
    N = dist.shape[0]

    # --- local observables ---
    n_i = np.full(N, filling)
    d_i = np.full(N, 0.25 * filling ** 2 / (1.0 + 0.35 * U))   # double occupancy

    # --- spin correlations ---
    xi = 1.0 + 2.0 / (1.0 + 0.3 * U)          # correlation length shrinks with U
    with np.errstate(divide="ignore"):
        env = np.exp(-dist / xi)

    # radial sign structure: NN negative, 2nd neighbour positive, alternating
    radial = np.cos(2 * np.pi * (dist - 0.5))

    # ANGULAR structure with the p6m symmetry of kagome.  This is what splits
    # pairs at the SAME distance into inequivalent sets (as really happens for
    # third neighbours on kagome: four antiferromagnetic + two ferromagnetic).
    # It is also what makes the distance-only baseline beatable -- without it
    # the benchmark is vacuous.
    theta = np.arctan2(rij[..., 1], rij[..., 0])
    angular = 0.35 * np.cos(6 * theta)

    C = -env * (radial + angular)
    C = 0.5 * (C + C.T)                        # correlations are symmetric

    # calibrate so the nearest-neighbour value approaches the known kagome
    # Heisenberg result  <S.S>_nn ~ -0.22  as U grows
    nn_mask = (dist > 0.4) & (dist < 0.6)
    target_nn = -0.22 * U / (U + 2.0)
    C *= target_nn / C[nn_mask].mean()

    C += 0.002 * rng.standard_normal(C.shape)  # measurement-like noise
    C = 0.5 * (C + C.T)

    # exact on-site identity  <S_i.S_i> = (3/4)(n_i - 2 d_i)   -- set LAST
    np.fill_diagonal(C, 0.75 * (n_i - 2 * d_i))
    return dict(U=U, filling=filling, L=L, n=n_i, d=d_i, C=C)


def make_dataset(Lx, Ly, n_samples, rng):
    out = []
    for i in range(n_samples):
        print("Making dataset: ", i+1, "/", n_samples)
        U = rng.uniform(0.5, 12.0)
        nu = rng.uniform(0.7, 1.0)
        t1 = rng.uniform(-0.5, 0.5)
        #out.append(synthetic_sample(L, U, nu, rng))
        current_time = time.time()
        out.append(solve_exact_diagonalization(Lx, Ly, U, nu, t1))
        total_time = time.time() - current_time
        print("\tTime: ", total_time)
    return out


# =============================================================================
# 3. THE MODEL
# =============================================================================

if HAVE_TORCH:

    class KagomeCNN(nn.Module):
        """Fully convolutional. NO dense layers -> works at any L.

        Input : per-site scalars packed as (B, F_in*3, L, L)
        Output: node embeddings h of shape (B, C, L, L, 3)
        """

        def __init__(self, f_in=4, width=64, depth=4):
            super().__init__()
            layers = []
            c_in = f_in * 3                       # 3 sublattices folded into channels
            for _ in range(depth):
                layers += [
                    nn.Conv2d(c_in, width * 3, kernel_size=3,
                              padding=1, padding_mode="circular"),  # periodic BC
                    nn.SiLU(),
                ]
                c_in = width * 3
            self.body = nn.Sequential(*layers)
            self.width = width

            # ---- node head: 1x1 conv, so still fully convolutional ----
            # predicts  [n_i, d_i]  only.  <S_i.S_i> is DERIVED, never predicted.
            self.node_head = nn.Conv2d(width * 3, 2 * 3, kernel_size=1)

            # ---- pair decoder: f(h_i, h_j, r_ij) -> C_ij ----
            # This replaces an N x N output layer, which would pin the size.
            self.pair = nn.Sequential(
                nn.Linear(2 * width + 3, 128), nn.SiLU(),
                nn.Linear(128, 128), nn.SiLU(),
                nn.Linear(128, 1),
            )

            # # ---- ADDED: kinetic pair decoder: f(h_i, h_j, r_ij) -> K_ij ----
            # self.kinetic_pair = nn.Sequential(
            #     nn.Linear(2 * width + 3, 128), nn.SiLU(),
            #     nn.Linear(128, 128), nn.SiLU(),
            #     nn.Linear(128, 1),
            # )

        def forward(self, x, rij, dist):
            """
            x    : (B, F_in*3, L, L)
            rij  : (B, N, N, 2) minimum-image displacements
            dist : (B, N, N)    distances
            """
            B, _, Lx, Ly = x.shape
            h = self.body(x)                              # (B, 3W, L, L)

            # ---- node observables ----
            node = self.node_head(h)                      # (B, 6, L, L) -> (B, 12, L, L)
            node = node.view(B, 2, 3, Lx, Ly)
            n_i = torch.sigmoid(node[:, 0]) * 2.0         # density in [0, 2]
            d_i = torch.sigmoid(node[:, 1])               # double occ in [0, 1]
            #CHANGE:
            #n_i = node[:, 0]
            #d_i = node[:, 1]
            #CHANGE2:
            n_i = torch.sigmoid(node[:, 0]) * 2.0          # density in [0, 2]
            d_fraction = torch.sigmoid(node[:, 1])         # fraction of max allowed occupancy
            d_i = d_fraction * (n_i / 2.0)                 # strictly <= n_i / 2, smooth gradients!***
            # physical constraint: d_i <= n_i/2
            #d_i = torch.minimum(d_i, n_i / 2)

            # ---- reshape embeddings to a flat site list ----
            hs = h.view(B, self.width, 3, Lx, Ly)           # (B, W, 3, L, L)
            hs = hs.permute(0, 3, 4, 2, 1).reshape(B, 3 * Lx * Ly, self.width)

            # ---- pair decoder over all pairs (Correlations) ----
            N = hs.shape[1]
            hi = hs[:, :, None, :].expand(B, N, N, self.width)
            hj = hs[:, None, :, :].expand(B, N, N, self.width)
            feat = torch.cat([hi, hj, rij, dist[..., None]], dim=-1)
            Cij = self.pair(feat).squeeze(-1)              # (B, N, N)
            Cij = 0.5 * (Cij + Cij.transpose(1, 2))        # symmetry: C_ij = C_ji

            # ---- ADDED: pair decoder over all pairs (Kinetic / Hopping terms) ----
            # Kij = self.kinetic_pair(feat).squeeze(-1)      # (B, N, N)
            # Kij = 0.5 * (Kij + Kij.transpose(1, 2))        # symmetry

            # ---- HARD-WIRED PHYSICS ----
            n_flat = n_i.permute(0, 2, 3, 1).reshape(B, N)
            d_flat = d_i.permute(0, 2, 3, 1).reshape(B, N)
            diag = 0.75 * (n_flat - 2 * d_flat)            # exact on-site identity
            Cij = Cij - torch.diag_embed(torch.diagonal(Cij, dim1=1, dim2=2))
            Cij = Cij + torch.diag_embed(diag)

            return dict(n=n_flat, d=d_flat, C=Cij) #K=Kij)


    # =========================================================================
    # 4. LOSS  --  multi-task with learned uncertainty weighting
    # =========================================================================

    class MultiTaskLoss(nn.Module):
        """Kendall, Gal & Cipolla (CVPR 2018): learn a per-task noise term
        instead of hand-tuning weights. Prevents the largest-scale target from
        swamping the small correlators, which is the usual failure mode."""

        def __init__(self, n_tasks=3):
            super().__init__()
            self.log_sigma = nn.Parameter(torch.zeros(n_tasks))

        def forward(self, losses):
            total = 0.0
            for k, Lk in enumerate(losses):
                total = total + torch.exp(-self.log_sigma[k]) * Lk + self.log_sigma[k]
            return total


    def correlation_loss(pred, target, eps=0.02):
        """Correlations span orders of magnitude and change sign, so plain MSE
        only ever learns the nearest-neighbour term. Weight by inverse magnitude
        so a 10% error on a small distant correlator counts like a 10% error on
        the dominant one."""
        w = 1.0 / (target.abs() + eps)
        return (w * (pred - target) ** 2).mean()


    def energy_from_observables(d, U, kinetic):
        """E = -t sum <c^dag c>  +  U sum <n_up n_dn>.
        Compute the energy, never predict it. Removes a head, removes a
        competing gradient, and gives an exact constraint for free."""
        return kinetic + U * d.sum(dim=1)


# =============================================================================
# 5. BASELINES  --  run these FIRST. If the CNN does not beat them, stop.
# =============================================================================

def _mse(pred_fn, data):
    return float(np.mean([np.mean((s["C"] - pred_fn(s)) ** 2) for s in data]))


def baseline_global_mean(train, test):
    """Level 0: one number for every entry. The floor."""
    mu = np.mean([s["C"].mean() for s in train])
    return _mse(lambda s: mu, test)


def baseline_mean_matrix(train, test):
    """Level 1: the sample-averaged matrix. Captures spatial structure but is
    blind to U."""
    Cbar = np.mean([s["C"] for s in train], axis=0)
    return _mse(lambda s: Cbar, test)


def baseline_distance_U(train, test, Lx, Ly, nd=20, nu=12):
    """Level 2: THE BASELINE THAT MATTERS.  Predicts C_ij from |r_i - r_j| AND
    U, i.e. everything except the angular / sublattice structure.  A model that
    cannot beat this has learned nothing that the radial profile does not
    already give away -- which is exactly the failure mode of a plain GNN."""
    _, dist = pair_displacements(Lx, Ly)
    dbins = np.linspace(0, dist.max() + 1e-9, nd + 1)
    Us = np.array([s["U"] for s in train])
    ubins = np.linspace(Us.min(), Us.max() + 1e-9, nu + 1)
    didx = np.clip(np.digitize(dist.ravel(), dbins) - 1, 0, nd - 1)

    table = np.zeros((nu, nd)); count = np.zeros((nu, nd))
    for s in train:
        u = np.clip(np.digitize(s["U"], ubins) - 1, 0, nu - 1)
        vals = s["C"].ravel()
        np.add.at(table[u], didx, vals)
        np.add.at(count[u], didx, 1.0)
    table /= np.maximum(count, 1.0)

    def pred(s):
        u = np.clip(np.digitize(s["U"], ubins) - 1, 0, nu - 1)
        return table[u][didx].reshape(dist.shape)
    return _mse(pred, test)


# =============================================================================
# 6. SANITY CHECKS  --  physics, not loss curves
# =============================================================================

def sanity_checks(sample, L, verbose=True):
    """Run these on EVERY model output. A good loss curve means nothing if
    these fail."""
    C = sample["C"]
    _, dist = pair_displacements(Lx, Ly)
    nn_mask = (dist > 0.4) & (dist < 0.6)          # nearest neighbours
    results = {
        "nn_correlation": C[nn_mask].mean(),        # target ~ -0.22 (Heisenberg)
        "sum_rule": C.sum(),                        # = 0 for a singlet
        "symmetry_violation": np.abs(C - C.T).max(),# must be ~0
        "diag_identity_error": np.abs(
            np.diag(C) - 0.75 * (sample["n"] - 2 * sample["d"])).max(),
    }
    if verbose:
        print("  nearest-neighbour <S.S>   : "
              f"{results['nn_correlation']:+.4f}   (Heisenberg limit ~ -0.22)")
        print(f"  sum rule  sum_ij <S_i.S_j>: {results['sum_rule']:+.4f}   "
              "(0 for a singlet; NOT satisfied by the")
        print("                              synthetic data above -- this check is"
              " for real ED data)")
        print(f"  |C - C^T|_max             : {results['symmetry_violation']:.2e}")
        print(f"  on-site identity error    : {results['diag_identity_error']:.2e}")
    return results
# =============================================================================
# ED ORACLE

#Helper: Generate 2D real-space coordinates for Kagome clusters
from quspin.basis import spinful_fermion_basis_1d
from quspin.operators import hamiltonian
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import quspin as qp

def get_kagome_positions(nx: int, ny: int) -> np.ndarray:
    """
    Generates (x, y) coordinates for an nx by ny Kagome lattice.
    Each unit cell contains 3 sites (base=0,1; tip=2).
    Total sites generated = nx * ny * 3.
    """
    positions = []
    
    for y in range(ny):
        for x in range(nx):
            # Kagome Bravais lattice shift vectors.
        
            cell_origin = y * A2 + x * A1   # matches site_positions: r*A2 + c*A1
            
            positions.append(cell_origin + BASIS[0])  # site 0 (A)
            positions.append(cell_origin + BASIS[1])  # site 1 (B)
            positions.append(cell_origin + BASIS[2])  # site 2 (C)
            
    return np.array(positions)

def get_kagome_bonds(Lx, Ly) -> list:
    """Helper function to find all bonds of a specific distance using Minimum Image Convention."""
    #Lx = L
    #Ly = L
    target_dist = [0.5, np.sqrt(3.0)/2.0]
    
    positions = get_kagome_positions(Lx, Ly)
    #num_sites = len(positions)
    num_sites = 3 * Lx * Ly
    
    #a1 = np.array([2.0, 0.0])
    #a2 = np.array([1.0, np.sqrt(3)])
    
    L1 = Lx * A1
    L2 = Ly * A2
    
    nx_range = [-1, 0, 1] if Lx > 1 else [0]
    ny_range = [-1, 0, 1] if Ly > 1 else [0]
    
    t1_bonds = []
    t2_bonds = []
    
    for i in range(num_sites):
        for j in range(i + 1, num_sites): 
            delta = positions[j] - positions[i]

            min_dist = float('inf')
            best_nx = 0
            best_ny = 0
            
            for nx in nx_range:
                for ny in ny_range:
                    shift = nx * L1 + ny * L2
                    dist = np.linalg.norm(delta + shift)
                    
                    if dist < min_dist:
                        min_dist = dist
                        best_nx = nx
                        best_ny = ny

            #CHANGE
            #t1_bonds.append((i, j))
            if np.isclose(min_dist , target_dist[0], atol=1e-4):
                t1_bonds.append((i, j))
            elif np.isclose(min_dist, target_dist[1], atol=1e-4):
                t2_bonds.append((i,j))
                
    return t1_bonds, t2_bonds
        
def build_fermi_hubbard_hamiltonian(Lx, Ly, U, filling, t, basis):
    """
    Builds a sparse SciPy matrix for the Fermi-Hubbard model on a Kagome sub-cluster.
    Restricts the Hilbert space to exactly N_up and N_down electrons.
    """
    
    # 1. Restrict the Hilbert Space
    # This single line drops the matrix size from 4^N down to (N C N_up) * (N C N_down)
    #Lx = L
    #Ly = L
    N_elec = 3*(Lx*Ly) * filling
    t1 = t
    t2 = 0.0
    #U = U
    theta_x = 0.0
    theta_y = 0.0
    bc_x = 'periodic'
    bc_y = 'periodic'
    
    #rij, dist = pair_displacements(L)
    #N = dist.shape[0] #num sites
    
    N_up = int(N_elec // 2 + (N_elec % 2))
    N_down = int(N_elec // 2)
    
    sites = 3 * Lx * Ly
    #basis = qp.basis.spinful_fermion_basis_1d(L=sites, Nf=(N_up, N_down))

    # 2. Get Lattice Bonds (You will need a helper function for your specific geometry)
    # These should be lists of tuples representing directional bonds: [(i, j), ...]
    # For Kagome, you only want to list each bond ONCE (e.g., i < j) because we use herm_con=True later.
    t1_bonds, t2_bonds = get_kagome_bonds(Lx, Ly)

    # 3. Format Interaction Lists for QuSpin
    # QuSpin uses operator strings where the left side of '|' is spin-up, right side is spin-down.
    
    # Hubbard U: n_{i,up} * n_{i,down}
    interaction_U = [[U, i, i] for i in range(sites)]

    # Hopping t: c^\dagger_i c_j 
    # '-' sign is included here because the physical hopping lowers the energy
    hop_up_t1 = [[-t1, i, j] for i, j in t1_bonds] + [[-t1, j, i] for i, j in t1_bonds]
    hop_dn_t1 = [[-t1, i, j] for i, j in t1_bonds] + [[-t1, j, i] for i, j in t1_bonds]
    
    hop_up_t2 = [[-t2, i, j] for i, j in t2_bonds] + [[-t2, j, i] for i, j in t2_bonds]
    hop_dn_t2 = [[-t2, i, j] for i, j in t2_bonds] + [[-t2, j, i] for i, j in t2_bonds]
    
    # Combine into QuSpin's static operator list
    static_operators = [
        ["n|n", interaction_U],     # U on both spins
        ["+-|", hop_up_t1],         # t1 hopping for spin-up
        ["|+-", hop_dn_t1],         # t1 hopping for spin-down
        ["+-|", hop_up_t2],         # t2 hopping for spin-up
        ["|+-", hop_dn_t2]          # t2 hopping for spin-down
    ]

    # 4. Build the Hamiltonian
    # FALSE herm_con=True automatically generates the Hermitian conjugates (the c^\dagger_j c_i terms), FALSE
    # meaning you don't have to manually write the reverse bonds.
    print("Forming Hamiltonian")
    H = hamiltonian(
        static_operators, 
        dynamic_list=[], 
        basis=basis, 
        dtype=np.complex128, 
        check_herm=False,
        check_pcon=False, # Skips a slow internal symmetry check
        check_symm=False  
    )

    # 5. Extract and return as standard SciPy CSR sparse matrix
    return H.tocsr()

def solve_exact_diagonalization(Lx, Ly, U, filling, t):
    """
    Solves exact quantum properties for small Kagome sub-clusters using SciPy sparse ED.
    Expects Lx, Ly, U, electron filling, t
    """
    # Step A: Parse input JSON string from Agent
    #changed inputs to directly get sites, t2, and U
    

    #Lx = L
    #Ly = L
    N_elec = 3*(Lx*Ly) * filling
    t1 = t
    t2 = 0.0
    #U = U
    theta_x = 0.0
    theta_y = 0.0
    bc_x = 'periodic'
    bc_y = 'periodic'
    
    rij, dist = pair_displacements(Lx, Ly)
    N = dist.shape[0] #num sites
    
    N_up = int(N_elec // 2 + (N_elec % 2))
    N_down = int(N_elec // 2)
    
    sites = 3 * Lx * Ly

    #TABC
    avg_energy = 0.0
    energy_zero = 0.0
    psi_zero = None
    params_zero = None
    average_observables = None
    
    #twists = [(0.0,0.0), (0.0,np.pi), (np.pi, 0.0), (np.pi, np.pi)]
    twists = [(0.0, 0.0)]
    for theta_x, theta_y in twists:
        #params['theta_x'] = theta_x
        #params['theta_y'] = theta_y
        # Step B: Build your sparse Kagome Hamiltonian H
        # (Note: Replace this stub with your actual Hamiltonian builder)
        num_qubits = 2 * sites
        #H_sparse = sp.csr_matrix((2**num_qubits, 2**num_qubits), dtype=np.complex128)]
        min_energy = 0.0
        best_k = 0
        best_gs_vector = None
        best_basis = None
        for k in range(Lx):
            print("Currently in: ", k)
            basis_k = qp.basis.spinful_fermion_basis_1d(
                L=sites, 
                Nf=(N_up, N_down),
                a=3,
                kblock=k
            )
            print("Basis formed")
            H_sparse = build_fermi_hubbard_hamiltonian(Lx, Ly, U, filling, t, basis_k)
            E_k, psi_k = spla.eigsh(H_sparse, k=1, which = "SA")
            print("Hamiltonian formed")
            if min_energy == 0.0:
                min_energy = E_k[0]
                best_gs_vector = psi_k[:,0].copy()
                best_basis = basis_k
                best_k = k
            if E_k[0] < min_energy:
                min_energy = E_k[0]
                best_gs_vector = psi_k[:,0].copy()
                best_basis = basis_k
                best_k = k
            del H_sparse, E_k, psi_k
            gc.collect()
            print('pass, ', k)
        #basis = qp.basis.spinful_fermion_basis_1d(L=sites, Nf=(N_up, N_down))
    
        # Step C: Diagonalize to find Ground State (k=1 smallest algebraic eigenvalue)
        #k_request = min(4, H_sparse.shape[0] - 2)
        #ground_eigenvalue, ground_eigenvector = spla.eigsh(
        #    H_sparse, k=1, which="SA"
            #H_sparse, k=3, which="SA"
            #H_sparse, k = k_request, which = 'SA'
        #)
        ground_eigenvalue = min_energy
        ground_eigenvector = best_gs_vector
        # Flatten state vector for 1D array operations
        ground_state = ground_eigenvector
        e_ground = float(np.real(ground_eigenvalue))
        #tolerance = 1e-8
        #degenerate_indices = np.where(np.abs(ground_eigenvalue - e_ground) < tolerance)[0]
        #degeneracy_count = len(degenerate_indices)
    
        # Step D: Get site geometry coordinates
        site_positions = get_kagome_positions(Lx, Ly)
    
        # Step E: Compute Observables via the helper function, NOTE: for all degeneracy states
        observables_n = compute_kagome_observables(
            ground_state = ground_state,
            num_sites = sites,
            Lx = Lx,
            Ly = Ly,
            site_positions = site_positions,
            basis = best_basis
        )

        if (theta_x, theta_y) == (0.0, 0.0):
            energy_zero = e_ground
            params_zero = {"Lx": Lx, "Ly":Ly, "U": U, "t1": t1, "t2": t2}
            psi_zero = ground_state.copy()
            observables_zero = observables_n.copy()

        if average_observables is None:
            average_observables = {k: np.array(v)/float(len(twists)) for k, v in observables_n.items()}
        else:
            for k in average_observables:
                average_observables[k] += np.array(observables_n[k]) /float(len(twists))

                
        avg_energy += e_ground/float(len(twists))
        
    # Step F: Assemble and Return Full Results Dict as JSON String
    output_payload = {
        "psi": psi_zero,
        #"model_params": params_zero,
        "energy": energy_zero,
        "average_energy": avg_energy,
        "observables": observables_zero#,
        #"average_observables": average_observables
    }
    
    n = np.array(observables_zero["charge_density"])
    d = np.array(observables_zero["double_occupancy"])
    C = np.array(observables_zero["spin_corr"])
    C2 = np.array(observables_zero["charge_corr"])
    B = np.array(observables_zero["B_matrix"])
    S_q_spin = np.array(observables_zero["S_q_spin"])
    S_q_charge = np.array(observables_zero["S_q_charge"])
    Adj1 = np.array(observables_zero["adj_1"])
    Adj2 = np.array(observables_zero["adj_2"])
    output = dict(t1=t1, t2=t2, U=U, filling=filling, Lx=Lx, Ly=Ly, n=n, d=d, C=C, C2=C2, B=B, Adj1=Adj1, Adj2=Adj2, energy=energy_zero, S_q_spin=S_q_spin, S_q_charge=S_q_charge)

    #return json.dumps(output_payload)
    return output
    


    
def compute_kagome_observables(ground_state: np.ndarray, num_sites: int, Lx: int, Ly: int, site_positions: np.ndarray, basis):
    """
    Computes Ground State Energy, Double Occupancy, Charge Correlation,
    Spin-Spin Correlation, and 2D Structure Factors for Kagome clusters
    within the constrained particle-conserving basis.
    
    Parameters:
        ground_state: Ground state eigenvector (dimension matches the constrained basis)
        num_sites: N (e.g., 3, 6, 9, 12)
        site_positions: Nx2 array of (x, y) coordinates for Kagome sites
        basis: The exact QuSpin basis object used to build the Hamiltonian
    """
    
    # 1 & 2. Build Sparse Number Operators n_i_up and n_i_down using QuSpin
    # This replaces the raw Pauli matrix approach and forces the operators 
    # to match the constrained dimension of the ground_state.
    n_up = []
    n_dn = []
    
    for i in range(num_sites):
        # Create constrained operators directly via QuSpin
        # "n|" means particle number operator for spin-up
        # "|n" means particle number operator for spin-down
        op_up = hamiltonian([["n|", [[1.0, i]]]], [], basis=basis, dtype=np.complex128, 
                            check_herm=False, check_symm=False, check_pcon=False)
        op_dn = hamiltonian([["|n", [[1.0, i]]]], [], basis=basis, dtype=np.complex128, 
                            check_herm=False, check_symm=False, check_pcon=False)
        
        # Convert to scipy sparse format so matrix math (@, +) works identically
        n_up.append(op_up.tocsr())
        n_dn.append(op_dn.tocsr())
        
    # 3. Double Occupancy Vector: <n_{i,up} * n_{i,dn}>
    double_occ = []
    for i in range(num_sites):
        D_op = n_up[i] @ n_dn[i]
        d_val = np.real(np.vdot(ground_state, D_op.dot(ground_state)))
        double_occ.append(float(d_val))

    #3.5 charge density:
    # Calculate expectation value: <psi | n | psi>
    #charge_density = []
    val_up = op_up.expt_value(ground_state).real
    val_dn = op_dn.expt_value(ground_state).real
        
    # Total charge at site i is the sum of up and down densities
    #charge_density.append(val_up + val_dn)
    charge_density = [
        np.real(np.vdot(ground_state, (n_up[i] + n_dn[i]).dot(ground_state)))
        for i in range(num_sites)
    ]

    # 4. Total Site Charge Operators N_i = n_{i,up} + n_{i,dn}
    N_op = [n_up[i] + n_dn[i] for i in range(num_sites)]
    
    # Charge Correlation Matrix (NxN)
    charge_corr = np.zeros((num_sites, num_sites))
    for i in range(num_sites):
        for j in range(num_sites):
            C_ij = N_op[i] @ N_op[j]
            charge_corr[i, j] = np.real(np.vdot(ground_state, C_ij.dot(ground_state)))

    # 5. Spin-Spin Correlation Matrix S_i . S_j = 3 * S^z_i S^z_j
    spin_corr = np.zeros((num_sites, num_sites))
    S_plus = []
    for i in range(num_sites):
        op = hamiltonian([["+|-", [[1.0, i, i]]]], [], basis=basis, dtype=np.complex128, check_herm=False, check_symm=False, check_pcon=False)
        S_plus.append(op.tocsr())
    S_minus = [Sp.conj().T for Sp in S_plus]

    for i in range(num_sites):
        S_z_i = 0.5 * (n_up[i] - n_dn[i])
        for j in range(num_sites):
            S_z_j = 0.5 * (n_up[j] - n_dn[j])
            #S_ij = (S_z_i @ S_z_j) + 0.5 * (S_plus[i] @ S_minus[j] + S_minus[i] @ S_plus[j])
            S_ij = 3.0 * (S_z_i @ S_z_j)
            spin_corr[i,j] = np.real(np.vdot(ground_state, S_ij.dot(ground_state)))

    # 6. 2D Spin Structure Factor S(q)
    # Example q-points in 2D Brillouin Zone
    momentum_density = 20
    qx = np.linspace(-np.pi, np.pi, momentum_density)
    qy = np.linspace(-np.pi, np.pi, momentum_density)
    QX, QY = np.meshgrid(qx, qy)
    
    # Flatten into a list of [qx, qy] coordinate pairs
    q_vectors = np.column_stack([QX.ravel(), QY.ravel()])

    # 6. 2D Spin Structure Factor S(q) over the dense grid
    spin_structure_factor = []
    for q in q_vectors:  # <--- Now iterating over the 400 grid points
        sq_val = 0.0
        for j in range(num_sites):
            for k in range(num_sites):
                r_jk = site_positions[j] - site_positions[k]
                phase = np.exp(-1j * np.dot(q, r_jk))
                sq_val += phase * spin_corr[j, k]
        spin_structure_factor.append(np.real(sq_val) / num_sites)
        
    # 7. 2D Charge Structure Factor S(q) over the dense grid
    charge_structure_factor = []
    for q in q_vectors:
        c_sq_val = 0.0
        for j in range(num_sites):
            for k in range(num_sites):
                r_jk = site_positions[j] - site_positions[k]
                phase = np.exp(-1j * np.dot(q, r_jk))
                c_sq_val += phase * charge_corr[j, k]
        charge_structure_factor.append(np.real(c_sq_val) / num_sites)

    # 8. Compute B_ij Matrix
    B_matrix = np.zeros((num_sites, num_sites))
    for i in range(num_sites):
        #diagonal of 1-particle density matrix is equal to the charge density
        B_matrix[i,i] =  charge_density[i]

        #calculate upper triangle and mirror (Bij == Bji)
        for j in range(i+1, num_sites):
            hoppings = [
                ["+-|", [[1.0, i, j], [1.0, j, i]]],
                ["|+-", [[1.0, i, j], [1.0, j, i]]]
            ]
            B_ij_op = hamiltonian(hoppings, [], basis=basis, dtype=np.complex128, 
                                  check_pcon=False, check_symm=False, check_herm=False)

            #compute expectation value:
            val = np.real(B_ij_op.expt_value(ground_state))
            #symmetrize matrix
            B_matrix[i,j] = val
            B_matrix[j,i] = val

    # 9. Adjacency matrix
    #L = int(np.sqrt(num_sites/3)) #assuming lattice shape is L*L e.g. 2x2 
    
    t1_bonds, t2_bonds = get_kagome_bonds(Lx, Ly)

    adj_1 = np.zeros((num_sites, num_sites))
    adj_2 = np.zeros((num_sites, num_sites))
    for i, j in t1_bonds:
        adj_1[i,j] = 1.0
        adj_1[j,i] = 1.0
    for i, j in t2_bonds:
        adj_2[i,j] = 1.0
        adj_2[j,i] = 1.0
    
    return {
        "double_occupancy": double_occ,
        "charge_corr": charge_corr.tolist(),
        "charge_density": charge_density,
        "spin_corr": spin_corr.tolist(),
        "S_q_spin": spin_structure_factor,
        "S_q_charge": charge_structure_factor,
        "q_vectors": q_vectors,
        "B_matrix": B_matrix.tolist(),
        "adj_1": adj_1.tolist(),
        "adj_2": adj_2.tolist()
    }

def make_dataset_from_folder(filepath, n_samples):
    data = []
    for i in range(n_samples):
        run_id = f"{i:03d}"
        h5_filepath = filepath + f"twelve_site_kagome_{run_id}.h5"
        with h5py.File(h5_filepath, 'r+') as f_h5:
            data_dict = {}
            for key in list(f_h5.keys()):
                data_dict[key] = f_h5[key][()]

        data.append(data_dict)
    return data
    
# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    rng = np.random.default_rng(0)
    #L = 3                                   # 2x2 unit cells = 12 sites (ED-friendly)
    Lx = 2
    Ly = 1
    N = 3 * Lx * Ly

    print("=" * 72)
    print(f"KAGOME GEOMETRY CHECK   Lx={Lx}, Ly={Ly} ->  {N} sites")
    print("=" * 72)
    adj = build_adjacency(Lx, Ly)
    degs = sorted({len(v) for v in adj.values()})
    print(f"  degrees present            : {degs}      (kagome must be 4)")
    tri = sum(1 for i in adj for j in adj[i] for k in adj[j]
              if k in adj[i] and i < j < k)
    print(f"  triangles                  : {tri}      (expect 2N/3 = {2*N//3})")

    print()
    print("=" * 72)
    print("STAGE B: DATA GENERATION & BASELINES")
    print("=" * 72)

    n_samples = 10
    # Generate exactly ONE datapoint for the overfitting test
    train = make_dataset(Lx, Ly, n_samples, rng) 
    sample = train[0]
    
    #train2 = []
    #train2.append(save_sample)
    #sample = train2[0]

    
    h5_path = f"data_outputs/quspin3/"
    train2 = make_dataset_from_folder(h5_path, n_samples)
    #train = train[70:90]
    #sample = train[0]
    for x in train2:
        print(list(x.keys()))
        train.append(x)
    
    # We will test on the identical point to set our target threshold
    try:
        b0 = baseline_global_mean(train, train)
        b1 = baseline_mean_matrix(train, train)
        b2 = baseline_distance_U(train, train, Lx, Ly)
    except:
        b0 = 0.0
        b1 = 0.0
        b2 = 0.0
    
    print(f"  L0 global mean             : {b0:.6f}")
    print(f"  L1 mean matrix (no U)      : {b1:.6f}")
    print(f"  L2 distance + U            : {b2:.6f}   <-- CNN MUST BEAT THIS MSE")

    print()
    print("=" * 72)
    print("SANITY CHECKS on the ED sample")
    print("=" * 72)
    sanity_checks(sample, Lx, Ly)

    if HAVE_TORCH:
        print()
        print("=" * 72)
        print("STAGE B: OVERFIT ONE DATAPOINT (DEBUGGING CODE/PHYSICS)")
        print("=" * 72)
        
        # 1. Format inputs for KagomeCNN
        U_val = sample["U"]
        fill_val = sample["filling"]
        #CHANGE
        t1_val = sample["t1"]
        t2_val = sample["t2"]
        
        # The CNN expects input shape (Batch, f_in*3, L, L). 
        # f_in is 2 (U and filling). 3 sublattices -> 6 channels total.
        #x = torch.zeros((1, 6, L, L), dtype=torch.float32)
        #x[:, 0::2, :, :] = float(U_val)     # Even channels: U
        #x[:, 1::2, :, :] = float(fill_val)  # Odd channels: filling
        #CHANGE
        #NEED to add channels for t1, t2
        x = torch.zeros((1, 12, Lx, Ly), dtype=torch.float32)

        x[:, 0::4, :, :] = float(U_val)     # Sublattice features for U
        x[:, 1::4, :, :] = float(fill_val)  # Sublattice features for filling
        x[:, 2::4, :, :] = float(t1_val)    # Sublattice features for t1
        x[:, 3::4, :, :] = float(t2_val)    # Sublattice features for t2
        
        rij, dist = pair_displacements(Lx, Ly)
        rij_t = torch.tensor(rij, dtype=torch.float32).unsqueeze(0)
        dist_t = torch.tensor(dist, dtype=torch.float32).unsqueeze(0)
        
        target_C = torch.tensor(sample["C"], dtype=torch.float32).unsqueeze(0)

        # 2. Setup Model & Optimizer
        #model = KagomeCNN(f_in=2, width=64, depth=4)
        #optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay = 0)

        file_path_model = "NN_Models/CNN_Model_4.pt"
        model = KagomeCNN(f_in=4, width=64, depth=4)
        try: 
            model.load_state_dict(torch.load(file_path_model, weights_only=True))
        except:
            print("model not found/model not compatible")

        target_C_obs = []
        target_n_obs = []
        target_d_obs = []

        for sample in train:
            target_C_i = sample["C"]
            target_C_obs.append(target_C_i)

            target_n_i = sample["n"]
            target_n_obs.append(target_n_i)

            target_d_i = sample["d"]
            target_d_obs.append(target_d_i) 

        # define MultiTaskLoss
        mtl = MultiTaskLoss(n_tasks=3)

        all_params = list(model.parameters()) + list(mtl.parameters())


        #CHANGE:
        # optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        optimizer = torch.optim.AdamW(all_params, lr=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50)
        
        # 3. Training Loop
        epochs = 300
        n_samples = 20
        print(f"Training for {epochs} epochs on a single datapoint...")
        model.train()
        for i in range(3):
            for j in range(n_samples):
                print(f"event: {i+1}, sample: {j+1}")
                sample = train[j]
                
                # 1. Format inputs for KagomeCNN
                U_val = sample["U"]
                fill_val = sample["filling"]
                t1_val = sample["t1"]
                t2_val = sample["t2"]
                try:
                    Lx = sample["Lx"]
                    Ly = sample["Ly"]
                except:
                    print("Lx, Ly not reassigned 1")
                    try:
                        Lx = sample["L"]
                        Ly = sample["L"]
                    except:
                        print("Lx, Ly not reassigned 1")
                        Lx = 2
                        Ly = 2
                    
            
                # The CNN expects input shape (Batch, f_in*3, L, L). 
                # f_in is 2 (U and filling). 3 sublattices -> 6 channels total.
                x = torch.zeros((1, 12, Lx, Ly), dtype=torch.float32)

                x[:, 0::4, :, :] = float(U_val)     # Sublattice features for U
                x[:, 1::4, :, :] = float(fill_val)  # Sublattice features for filling
                x[:, 2::4, :, :] = float(t1_val)    # Sublattice features for t1
                x[:, 3::4, :, :] = float(t2_val)    # Sublattice features for t2
    
                # --- ADDED: Extract target kinetic terms from sample ---
                # Assuming your sample dictionary stores the bond kinetic terms as "K" or similar
                # for key, value in sample.items():
                #     print(f"Key: {key} (type: {type(key).__name__}), Value: {value} (type: {type(value).__name__})")
                rij, dist = pair_displacements(Lx, Ly)
                rij_t = torch.tensor(rij, dtype=torch.float32).unsqueeze(0)
                dist_t = torch.tensor(dist, dtype=torch.float32).unsqueeze(0)
    
                target_C = torch.tensor(sample["C"], dtype=torch.float32).unsqueeze(0)
                target_n = torch.tensor(sample["n"], dtype=torch.float32).unsqueeze(0)
                target_d = torch.tensor(sample["d"], dtype=torch.float32).unsqueeze(0)
                #target_K = torch.tensor(sample["K"], dtype=torch.float32).unsqueeze(0)
                
                #mean and std
                #C_mean, C_std = target_C.mean(), target_C.std()
                #n_mean, n_std = target_n.mean(), target_n.std()
                #d_mean, d_std = target_d.mean(), target_d.std()
    
                #print(f"std_n: {n_std}\n std: {d_std}")
    
                for ep in range(epochs):
                    optimizer.zero_grad()
                    
                    # Forward pass
                    out = model(x, rij_t, dist_t)
                    pred_C = out["C"]
                    #pred_K = out["K"]
                    pred_n = out["n"]
                    pred_d = out["d"]
                    
    
                    assert pred_C.shape == target_C.shape
                    #assert pred_K.shape == target_K.shape
                    assert pred_n.shape == target_n.shape
                    assert pred_d.shape == target_d.shape
    
    
                    #norm_pred_C = (pred_C - C_mean) / C_std
                    #norm_target_C = (target_C - C_mean) / C_std
    
                    #norm_pred_n = (pred_n - n_mean) / n_std
                    #norm_target_n = (target_n - n_mean) / n_std
    
                    #norm_pred_d = (pred_d - d_mean) / d_std
                    #norm_target_d = (target_d - d_mean) / d_std
    
    
                    # --- ADDED: Combined Loss for Correlators and Kinetic Terms ---
                    loss_C = F.mse_loss(pred_C, target_C)
                    loss_n = F.mse_loss(pred_n, target_n)
                    # #loss_K = F.mse_loss(pred_K, target_K)
                    loss_d = F.mse_loss(pred_d, target_d)
    
                    # loss_C = F.mse_loss(norm_pred_C, norm_target_C)
                    # loss_n = F.mse_loss(norm_pred_n, norm_target_n)
                    # loss_d = F.mse_loss(norm_pred_d, norm_target_d)
    
                    # loss_d = scaled_mse_loss(pred_d, target_d, scale_factor=4.0)
                    # loss_n = scaled_mse_loss(pred_n, target_n, scale_factor=0.5)
                    # loss_C = F.mse_loss(pred_C, target_C) 
    
                    # Total loss balancing or multi-task weighting
                    # loss = loss_C + loss_n + loss_d  #+ loss_K   
    
                    #mutlitaskloss 
                    loss = mtl([loss_C, loss_d, loss_n])
                    '''
                    if epochs < 200:
                        loss = loss_C
                    elif epochs < 400:
                        loss = loss_d
                    else:
                        loss = loss_n
                    '''
    
                    # test for loss: heavily weighting C's loss 
                    #loss = (10.0 * loss_C) + (1.0 * loss_d) + (1.0 * loss_n)
    
    
    
                    # Standard MSE Loss matching the baseline metric
                    # loss = F.mse_loss(pred_C, target_C)
                    
    
                    # --- ADDED: Derived Energy Calculation (for monitoring/physics checks) ---
                    # E = -t * sum(<c^+ c>) + U * sum(<n^dagger n>)
                    # Summing over the bond network / sites
    
    
                    # kinetic_energy_total = -t_val * torch.sum(pred_K)
                    # interaction_energy_total = float(U_val) * torch.sum(pred_d)
                    # derived_total_energy = kinetic_energy_total + interaction_energy_total
    
                    # Backward pass
                    loss.backward()
                    optimizer.step()
        
                    # --- DEBUGGING BLOCK: Paste this after loss.backward() ---
        
                    # 1. Check Precision & Dtypes
                    #if ep == 1:
                    #    print("-" * 50)
                    #    print(f"  Epoch {ep+1:03d}/{epochs} | MSE Loss: {loss.item():.6f}")
                        #print(f"\n--- PRECISION CHECK ---")
                        #print(f"pred_C dtype: {pred_C.dtype} | target_C dtype: {target_C.dtype}")
                        #print(f"Model dtype: {next(model.parameters()).dtype}")
                    
                    # 2. Check Target vs Prediction Boundaries (Is Sigmoid artificially capping the output?)
                    #print(f"Epoch {ep:03d} | Pred Min/Max: {pred_C.min().item():.5f} to {pred_C.max().item():.5f}")
                    #print(f"Epoch {ep:03d} | Targ Min/Max: {target_C.min().item():.5f} to {target_C.max().item():.5f}")
                    
                    # 3. Check for Vanishing Gradients (Underflow or Saturation)
                    max_grad = 0.0
                    for name, p in model.named_parameters():
                        if p.grad is not None:
                            grad_max = p.grad.abs().max().item()
                            if grad_max > max_grad:
                                max_grad = grad_max
                            # Optional: uncomment to find exactly which layer has dead gradients
                            # if grad_max < 1e-7: 
                            #     print(f"Dead gradient in {name}: {grad_max}")
                    
                    #print(f"Epoch {ep:03d} | Max Gradient magnitude: {max_grad:.2e}")
                    #print("-" * 50)
                    # --------------------------------------------------------
                                
                    if (ep + 1) % 300 == 0 or ep == 0:
                    #if ep > -1:
                        print("-" * 50)
                        print(f"  Epoch {ep+1:03d}/{epochs} | Total Loss: {loss.item():.6f}")
                        # print(f"  Sub-losses -> C: {loss_C.item():.6f}, n: {loss_n.item():.6f}, d: {loss_d.item():.6f}")
                        # print(f"  Derived Total Energy: {derived_total_energy.item():.4f}")
                        print(f"  Sub-losses -> C: {loss_C.item():.6f}, n: {loss_n.item():.6f}, d: {loss_d.item():.6f}")
                        #for name, param in model.named_parameters():
                        #    if param.grad is not None:
                        #        print(f"{name} grad norm: {param.grad.norm().item()}")
                    
        
                print("-" * 72)
                loss2 = loss_C
                final_mse = loss2.item()
                print(f"Final Model MSE  : {final_mse:.6f}")
                print(f"L2 Baseline MSE  : {b2:.6f}")
                #print(f"diff charge density = {abs(pred_n - target_n)}")
                #print(f"diff double occupancy = {abs(pred_d - target_d)}")
                
                if final_mse < b2:
                    print(">> SUCCESS: The CNN successfully overfit the data and beat the spatial baseline.")
                    #print(">> CONCLUSION: Architecture and physics embedding (pair decoder) are working correctly.")
                else:
                    print(">> FAIL: The CNN could not overfit a single point better than the baseline.")
                    #print(">> CONCLUSION: There is likely a bug in the model architecture, feature packing, or gradients.")

KAGOME GEOMETRY CHECK   Lx=2, Ly=1 ->  6 sites
  degrees present            : [3, 4]      (kagome must be 4)
  triangles                  : 4      (expect 2N/3 = 4)

STAGE B: DATA GENERATION & BASELINES
Making dataset:  1 / 10
Currently in:  0
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass,  0
Currently in:  1
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass,  1
	Time:  0.4788327217102051
Making dataset:  2 / 10
Currently in:  0
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass,  0
Currently in:  1
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass,  1
	Time:  0.4816560745239258
Making dataset:  3 / 10
Currently in:  0
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass,  0
Currently in:  1
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass,  1
	Time:  0.48920512199401855
Making dataset:  4 / 10
Currently in:  0
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass,  0
Currently in:  1
Basis formed
Forming Hamiltonian
Hamiltonian formed
pass